In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/colab/placement_predict_50k (5).csv')
df.head()

,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Certifications,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly
0,1,Female,Delhi,Tier3,IT,DataScience,Yes,Yes,6.39,6.84,...,1,0,62.3,6.57,40.6,65.7,No,Medium,0,0
1,2,Male,Chennai,Tier2,ECE,AI,Yes,No,5.95,6.74,...,2,0,44.0,5.86,40.3,51.8,No,Medium,0,0
2,3,Female,Hyderabad,Tier3,ECE,Networking,No,No,7.13,8.11,...,2,1,73.8,7.50,73.6,67.9,No,High,1,0
3,4,Female,Jaipur,Tier3,ECE,Embedded,No,No,9.37,9.97,...,6,2,100.0,9.41,98.7,NaN,No,Excellent,1,0
4,5,Male,Ahmedabad,Tier3,Mechanical,DataScience,No,No,8.25,8.99,...,4,2,90.8,9.24,83.1,100.0,No,Excellent,1,0


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

In [4]:
df = pd.read_csv('/content/drive/MyDrive/colab/placement_predict_50k (5).csv')
print('Dataset Shape:', df.shape)
print('Columns:', df.columns.tolist())

# Target: 0 = Not Placed, 1 = Placed
y = df['PlacementStatus']

features = [
    'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel',
    'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4',
    'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA',
    'AttendancePercent', 'Internships', 'Projects', 'Workshops',
    'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating',
    'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular'
]

X = df[features].copy()

categorical_features = [
    'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation',
    'Hostel', 'HistoryOfBacklogs', 'ExtraCurricular'
]

numerical_features = [c for c in features if c not in categorical_features]

for col in numerical_features:
    X.loc[:, col] = X[col].fillna(X[col].median())
for col in categorical_features:
    X.loc[:, col] = X[col].fillna(X[col].mode()[0])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
])

Dataset Shape: (50000, 31)
Columns: ['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']


In [5]:
l1_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=1.0,
        max_iter=2000,
        random_state=42,
    ))
])

l1_model.fit(X_train, y_train)
y_pred_l1 = l1_model.predict(X_test)
y_prob_l1 = l1_model.predict_proba(X_test)[:, 1]

print('\nL1 Logistic Regression')
print('Accuracy:', round(accuracy_score(y_test, y_pred_l1), 4))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_l1))
print('Classification Report:')
print(classification_report(y_test, y_pred_l1, target_names=['Not Placed', 'Placed']))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob_l1), 4))


L1 Logistic Regression
Accuracy: 0.7903
Confusion Matrix:
[[4227 1023]
 [1074 3676]]
Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000

ROC-AUC: 0.8769


In [6]:
l2_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        C=1.0,
        max_iter=2000,
        random_state=42,
    ))
])

l2_model.fit(X_train, y_train)
y_pred_l2 = l2_model.predict(X_test)
y_prob_l2 = l2_model.predict_proba(X_test)[:, 1]

print('\nL2 Logistic Regression')
print('Accuracy:', round(accuracy_score(y_test, y_pred_l2), 4))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_l2))
print('Classification Report:')
print(classification_report(y_test, y_pred_l2, target_names=['Not Placed', 'Placed']))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob_l2), 4))


L2 Logistic Regression
Accuracy: 0.7907
Confusion Matrix:
[[4229 1021]
 [1072 3678]]
Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000

ROC-AUC: 0.8769


In [7]:
# Elastic regularization
elastic_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,
        C=1.0,
        max_iter=5000,
        random_state=42,
    ))
])

elastic_model.fit(X_train, y_train)
y_pred_elastic = elastic_model.predict(X_test)
y_prob_elastic = elastic_model.predict_proba(X_test)[:, 1]

print('\nElastic Logistic Regression')
print('Accuracy:', round(accuracy_score(y_test, y_pred_elastic), 4))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_elastic))
print('Classification Report:')
print(classification_report(y_test, y_pred_elastic, target_names=['Not Placed', 'Placed']))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob_elastic), 4))


Elastic Logistic Regression
Accuracy: 0.7905
Confusion Matrix:
[[4228 1022]
 [1073 3677]]
Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000

ROC-AUC: 0.8769
